# World Cup Sentiment Tracker

## Data Exploration **(Real Tweets)** 

---

### First Dataset:

Source type:  
Replies/comments under official football tweets  
(less likely to be biased and/or noisy since official posts are usually neutral/promotional, but the replies are fan reactions)

Accounts:
- @FIFAWorldCup
- national team accounts
- major player/team accounts if relevant

Tweet types:
- goal announcements
- full-time result posts
- VAR/penalty/red-card posts
- lineup posts
- highlight clips

Collected unit:  
Individual reply/comment, not the official post itself

----
### Strategy:

1. Pick one match/event
2. Find official posts about that event
3. Pull replies under those posts
4. Store replies with metadata
5. Run baseline sentiment
6. Manually inspect model failures

---

## Real Football Tweet Collection

Goal:
Collect replies from official football event posts and evaluate how well the baseline sentiment model performs on real football fan language.

Questions:
1. Can we successfully collect real football discussion?
2. Does the baseline sentiment model perform reasonably well?
3. What types of football language cause model failures?

In [ ]:
import sys
sys.path.append("../src")

import pandas as pd

from api.getxapi_client import (
    fetch_replies,
    normalize_replies
)
from data.match_linking import (
    link_replies_to_matches,
    load_match_schedule,
)
from data.preprocess import (
    annotate_replies,
    preprocess_reply_buckets,
    preprocess_replies,
    preprocessing_summary
)
from models.sentiment_baseline import (
    add_sentiment,
    load_sentiment_model
)

In [4]:
response = fetch_replies(tweet_id="2069468618476200189")

response.keys()

dict_keys(['tweetId', 'reply_count', 'has_more', 'next_cursor', 'replies'])

In [5]:
rows = normalize_replies(
    response=response,
    match="Portugal vs Spain",
    event="goal",
    source_account="FIFAWorldCup",
    team="Portugal"
)

raw_df = pd.DataFrame(rows)

raw_df.head()

,tweet_id,parent_tweet_id,url,timestamp,text,author_username,author_name,like_count,reply_count,view_count,match,team,player,event,source_account,source
0,2069547886975635584,2069468618476200189,https://x.com/NoCaptionMood/status/20695478869...,Tue Jun 23 22:27:21 +0000 2026,@FIFAWorldCup https://x.com/i/status/206952949...,NoCaptionMood,No Caption Mood,71,1,20260,Portugal vs Spain,Portugal,None,goal,FIFAWorldCup,getxapi
1,2069547912825049173,2069468618476200189,https://x.com/grok/status/2069547912825049173,Tue Jun 23 22:27:27 +0000 2026,@NoCaptionMood @FIFAWorldCup Ask Grok is curre...,grok,Grok,111,1,11462,Portugal vs Spain,Portugal,None,goal,FIFAWorldCup,getxapi
2,2069898353811521844,2069468618476200189,https://x.com/STEMCELLTech/status/206989835381...,Wed Jun 24 21:39:59 +0000 2026,Enter your best cell image by 11:59 a.m. PDT o...,STEMCELLTech,STEMCELL Technologies,4,0,314107,Portugal vs Spain,Portugal,None,goal,FIFAWorldCup,getxapi
3,2069472339855556889,2069468618476200189,https://x.com/Shuni_Buccaneer/status/206947233...,Tue Jun 23 17:27:09 +0000 2026,@FIFAWorldCup Messi's fans right now🤣🤣🤣 https:...,Shuni_Buccaneer,Shuni_We_Buccaneer☠️,639,15,23595,Portugal vs Spain,Portugal,None,goal,FIFAWorldCup,getxapi
4,2069488045565374770,2069468618476200189,https://x.com/TeeMacKuz/status/206948804556537...,Tue Jun 23 18:29:33 +0000 2026,@FIFAWorldCup When Ronaldo scores\nThe whole w...,TeeMacKuz,Oluwa Loba,553,8,14542,Portugal vs Spain,Portugal,None,goal,FIFAWorldCup,getxapi


In [6]:
raw_df.columns

Index(['tweet_id', 'parent_tweet_id', 'url', 'timestamp', 'text',
       'author_username', 'author_name', 'like_count', 'reply_count',
       'view_count', 'match', 'team', 'player', 'event', 'source_account',
       'source'],
      dtype='str')

In [7]:
raw_df.to_csv(
    "../data/raw/real_replies_sample.csv",
    index=False
)

In [8]:
raw_df.shape

(39, 16)

In [9]:
raw_df["text"].head(20)

0     @FIFAWorldCup https://x.com/i/status/206952949...
1     @NoCaptionMood @FIFAWorldCup Ask Grok is curre...
2     Enter your best cell image by 11:59 a.m. PDT o...
3     @FIFAWorldCup Messi's fans right now🤣🤣🤣 https:...
4     @FIFAWorldCup When Ronaldo scores\nThe whole w...
5     @FIFAWorldCup @pmcafrica THE GREATEST OF ALL T...
6     @FIFAWorldCup Another record set! https://t.co...
7     @FIFAWorldCup No one—I repeat, no one can ever...
8     @FIFAWorldCup Cristiano Ronaldo https://t.co/f...
9            @FIFAWorldCup GOAT https://t.co/JlKMs9LZxe
10    @FIFAWorldCup Another record https://t.co/u32B...
11    ¡SUFRIDO TRIUNFO CROATA!\n\nAunque por la míni...
12    @FIFAWorldCup @BigDee_UTD https://t.co/oXdQR8BwAk
13    @FIFAWorldCup Is he back to being the goat again?
14                                    @FIFAWorldCup 🐐🐐🐐
15      @FIFAWorldCup Don't ever doubt the goat again !
16    @FIFAWorldCup SUIII. Six World Cups. One Crist...
17                         @FIFAWorldCup Cr7 the

In [ ]:
reply_buckets = preprocess_reply_buckets(raw_df)

annotated_df = reply_buckets["annotated"]

schedule_df = load_match_schedule("../data/reference/matches_sample.csv")
annotated_df = link_replies_to_matches(
    annotated_df,
    schedule_df
)

analysis_df = annotated_df[annotated_df["filter_reason"] == "keep"].reset_index(drop=True)
review_df = analysis_df[analysis_df["needs_context_review"]].reset_index(drop=True)
removed_df = annotated_df[annotated_df["filter_reason"] != "keep"].reset_index(drop=True)

preprocessing_summary(annotated_df, analysis_df)

In [11]:
pd.set_option("display.max_colwidth", None)

review_df[
    [
        "text",
        "clean_text",
        "parent_context_confidence",
        "contextual_terms",
        "context_relevance_boost",
        "needs_context_review",
        "relevance_score"
    ]
].head(20)

,text,clean_text,parent_context_confidence,contextual_terms,context_relevance_boost,needs_context_review,relevance_score
0,@FIFAWorldCup @pmcafrica THE GREATEST OF ALL TIME https://t.co/P6KOwPDIUK,THE GREATEST OF ALL TIME,3,[],1,True,1
1,@FIFAWorldCup Another record set! https://t.co/oCaTMNXXjZ,Another record set!,3,"[another record, record]",2,True,2
2,"@FIFAWorldCup No one—I repeat, no one can ever achieve what this man has achieved https://t.co/ZDXVNGg2aL","No one—I repeat, no one can ever achieve what this man has achieved",3,"[achieve, achieved, man, no one, this man]",2,True,2
3,@FIFAWorldCup Another record https://t.co/u32Bczokqt,Another record,3,"[another record, record]",2,True,2
4,Bring yourself and your phone to where the soft life is. Say bye-bye to your old contract and get up to $800 per line.,Bring yourself and your phone to where the soft life is. Say bye-bye to your old contract and get up to $800 per line.,3,[],1,True,1
5,@FIFAWorldCup haters in disbelief https://t.co/LRTC0Sy5kv,haters in disbelief,3,[],1,True,1
6,@FIFAWorldCup The G.O.A.T,The G.O.A.T,3,[],1,True,1
7,@FIFAWorldCup Built different!🔥,Built different!🔥,3,[],1,True,1
8,@FIFAWorldCup @Nanakwamecriss But they said we are finished!!!,But they said we are finished!!!,3,[],1,True,1


In [ ]:
analysis_df[
    [
        "text",
        "clean_text",
        "matched_entities",
        "inferred_teams",
        "inferred_players",
        "inferred_managers",
        "linked_match_id",
        "linked_match",
        "linked_match_confidence",
        "linked_match_method",
        "entity_confidence",
        "parent_context_confidence",
        "contextual_terms",
        "context_relevance_boost",
        "needs_context_review",
        "relevance_score",
        "filter_reason"
    ]
].head(20)

In [13]:
classifier = load_sentiment_model()

analysis_df = add_sentiment(
    analysis_df,
    text_column="clean_text",
    classifier=classifier
)

analysis_df[
    [
        "text",
        "clean_text",
        "sentiment_label",
        "sentiment_score",
        "relevance_score",
        "needs_context_review"
    ]
].head(20)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

,text,clean_text,sentiment_label,sentiment_score,relevance_score,needs_context_review
0,Enter your best cell image by 11:59 a.m. PDT on June 29 for a chance to win prizes! \n\nThe 20 top-voted images will receive a magnetic puzzle of their image. 🧩,Enter your best cell image by 11:59 a.m. PDT on June 29 for a chance to win prizes! The 20 top-voted images will receive a magnetic puzzle of their image. 🧩,POSITIVE,0.999131,4,False
1,@FIFAWorldCup Messi's fans right now🤣🤣🤣 https://t.co/FcFd4StXrT,Messi's fans right now🤣🤣🤣,POSITIVE,0.995415,3,False
2,@FIFAWorldCup When Ronaldo scores\nThe whole world talks about Ronaldo \n\nWhen Messi scores\nThe whole world talk about Ronaldo \n\nThe Real GOAT.,When Ronaldo scores The whole world talks about Ronaldo When Messi scores The whole world talk about Ronaldo The Real GOAT.,POSITIVE,0.997308,8,False
3,@FIFAWorldCup @pmcafrica THE GREATEST OF ALL TIME https://t.co/P6KOwPDIUK,THE GREATEST OF ALL TIME,POSITIVE,0.999836,1,True
4,@FIFAWorldCup Another record set! https://t.co/oCaTMNXXjZ,Another record set!,POSITIVE,0.998892,2,True
5,"@FIFAWorldCup No one—I repeat, no one can ever achieve what this man has achieved https://t.co/ZDXVNGg2aL","No one—I repeat, no one can ever achieve what this man has achieved",NEGATIVE,0.997070,2,True
6,@FIFAWorldCup Cristiano Ronaldo https://t.co/fANSpg4b6I,Cristiano Ronaldo,POSITIVE,0.998823,7,False
7,@FIFAWorldCup GOAT https://t.co/JlKMs9LZxe,GOAT,NEGATIVE,0.973965,3,False
8,@FIFAWorldCup Another record https://t.co/u32Bczokqt,Another record,POSITIVE,0.990748,2,True
9,"¡SUFRIDO TRIUNFO CROATA!\n\nAunque por la mínima, Croacia consiguió la victoria ante Panama, quienes quedaron eliminados del Mundial 2026.\n\nNi te pierdas las mejores jugadas https://t.co/EV5o3N1pMr","¡SUFRIDO TRIUNFO CROATA! Aunque por la mínima, Croacia consiguió la victoria ante Panama, quienes quedaron eliminados del Mundial 2026. Ni te pierdas las mejores jugadas",NEGATIVE,0.964972,5,False


In [ ]:
sample = analysis_df.sample(
    min(25, len(analysis_df)),
    random_state=42
)

sample[
    [
        "text",
        "clean_text",
        "matched_entities",
        "inferred_teams",
        "inferred_players",
        "linked_match_id",
        "linked_match",
        "linked_match_confidence",
        "needs_context_review",
        "sentiment_label",
        "sentiment_score",
        "relevance_score"
    ]
]

In [15]:
annotated_df.to_csv(
    "../data/processed/annotated_replies_sample.csv",
    index=False
)

analysis_df.to_csv(
    "../data/processed/analysis_replies_loose.csv",
    index=False
)

review_df.to_csv(
    "../data/processed/review_replies.csv",
    index=False
)

removed_df.to_csv(
    "../data/processed/removed_replies.csv",
    index=False
)

In [ ]:
summary = {
    "raw_rows": len(raw_df),
    "annotated_rows": len(annotated_df),
    "analysis_rows": len(analysis_df),
    "review_rows": len(review_df),
    "removed_rows": len(removed_df),
    "linked_analysis_rows": analysis_df["linked_match_id"].notna().sum(),
    "ambiguous_analysis_rows": (analysis_df["linked_match_confidence"] == "ambiguous").sum(),
    "retention_rate": len(analysis_df) / len(raw_df)
}

summary

### Observations

Real reply data contains media-only, text-poor, spam-like, low-relevance, context-dependent, and match-ambiguous replies. The current preprocessing is intentionally recall-oriented: it keeps more potentially relevant football discussion, including vague replies that rely on parent tweet context, while accepting that some junk can still slip into `analysis_df`.

The working pipeline is now:

```text
Raw API replies
-> normalized rows
-> clean text
-> football entity detection
-> parent-context relevance boost
-> inferred team/player/manager metadata
-> schedule-based match linking
-> data quality filtering
-> baseline sentiment labels and scores
```

`analysis_df` is the dataset sent to the baseline sentiment model. `review_df` is the subset of kept rows that rely mainly on parent context and should be manually inspected. `removed_df` keeps clear junk and unusable text for audit.

Match linking is conservative. It uses inferred teams, known parent context, and reply or parent timestamps to choose a scheduled match only when the evidence is strong enough. Rows can remain unlinked or ambiguous through `linked_match_confidence`, which is preferable to forcing bad match labels.

This loose filter is acceptable for the MVP because it prioritizes retaining real fan reactions over perfectly excluding every irrelevant reply. A future production version could add a second-stage relevance classifier or embedding similarity check to reduce false positives, and replace `matches_sample.csv` with a full official fixture schedule.

Filtering and enrichment are implemented in `src/data/preprocess.py`, `src/data/entities.py`, and `src/data/match_linking.py`, with reference data in `data/reference/football_entities.json` and `data/reference/matches_sample.csv`. Sentiment scoring is applied with the baseline DistilBERT model through `src/models/sentiment_baseline.py`.